# **[_Python Auto Loader_](url)**

In [0]:
%sql
SELECT current_catalog() as catalog, current_schema() as schema;

In [0]:
%sql
-- Change the current catalog and schema
USE CATALOG pysaprk_demo;
USE SCHEMA autoloader;

-- List the current catalog and schema
SELECT current_catalog() as catalog, current_schema() as schema;

In [0]:
## Create a volumn in the current database
spark.sql(f"CREATE VOLUME IF NOT EXISTS {spark.catalog.currentCatalog()}.{spark.catalog.currentDatabase()}.auto_loader_files")

## Set checkpoint location to the volume from above
checkpoint_file_location = f"/Volumes/{spark.catalog.currentCatalog()}/{spark.catalog.currentDatabase()}/auto_loader_files/checkpoint/"

## Set the location of the incoming data
data_location = f"/Volumes/{spark.catalog.currentCatalog()}/{spark.catalog.currentDatabase()}/auto_loader_files/data"

In [0]:
## Incremental (or stream) data using auto loader
checkpoint_path = f"/Volumes/{spark.catalog.currentCatalog()}/{spark.catalog.currentDatabase()}/auto_loader_files/checkpoint/"

(
  spark
  .readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("sep", ",")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load(data_location)
  .writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(once=True)
  .toTable(f"{spark.catalog.currentCatalog()}.{spark.catalog.currentDatabase()}.python_csv_autoloader")
)

In [0]:
## Incremental (or stream) data using auto loader
checkpoint_path = f"/Volumes/{spark.catalog.currentCatalog()}/{spark.catalog.currentDatabase()}/auto_loader_files/checkpoint/"

(
  spark
  .readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("sep", ",")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load(data_location)
  .writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(once=True)
  .toTable(f"{spark.catalog.currentCatalog()}.{spark.catalog.currentDatabase()}.python_csv_autoloader")
)

In [0]:
%sql
DESCRIBE HISTORY python_csv_autoloader;

In [0]:
%sql
DESCRIBE EXTENDED python_csv_autoloader

In [0]:
%sql
DROP TABLE IF EXISTS python_csv_autoloader;